# Model Implementation

Check the implementation of various models

# Data

In [1]:
import altair as alt
import numpy as np
import polars as pl
import torch

from ts_uncertainty.plotting import plot_synthetic_sample
from ts_uncertainty.synthetic import SyntheticGenerator, SyntheticSample

alt.renderers.enable("jupyter", offline=True)

RendererRegistry.enable('jupyter')

In [6]:
context_length = 48
horizon_length = 12

sin_gen = SyntheticGenerator(
    context_length=context_length,
    horizon=horizon_length,
    seasonal=True,
    periods=(24,),             # period of the sine
    n_periods=1,
    n_harmonics=1,             # 1 harmonic = pure sine, no extra shape
    level_range=(0.0, 0.0),    # centered at 0
    slope_range=(0.0, 0.0),    # no trend
    amplitude_range=(1.0, 1.0),
    noise=False,
    spikes=False,
    shift=False,
)
s = sin_gen.sample()
plot_synthetic_sample(s, title="Sin Sample")

In [7]:
# model input and output shapes
x = torch.tensor(s.context).reshape(1, -1, 1)  # (B, T, V)
y = torch.tensor(s.target).reshape(1, -1, 1)  # (B, T, V)

## LSTM

In [8]:
from ts_uncertainty.models.lstm import LSTM_Model

In [9]:
hidden_size = 32
ref_model = LSTM_Model(
    input_size=1,
    hidden_size=hidden_size,
    horizon_length=horizon_length,
    target_size=1,
    use_torch = True,
    init_constant = True,
)
my_model = LSTM_Model(
    input_size=1,
    hidden_size=hidden_size,
    horizon_length=horizon_length,
    target_size=1,
    use_torch = False,
    init_constant = True,
)

print("Weights and biases are equal between the two models:")
print(torch.allclose(ref_model.lstm.weight_ih_l0, my_model.lstm.weight_ih))
print(torch.allclose(ref_model.lstm.weight_hh_l0, my_model.lstm.weight_hh))
print(torch.allclose(ref_model.lstm.bias_ih_l0, my_model.lstm.bias_ih))
print(torch.allclose(ref_model.lstm.bias_hh_l0, my_model.lstm.bias_hh))
print("Output of the two models are equal:")
print(torch.allclose(ref_model(x), my_model(x)))


Weights and biases are equal between the two models:
True
True
True
True
Output of the two models are equal:
True


## Transformer